# Stage 4.5 v4 — TrackNet ball detector (Colab GPU training)

**Runtime → Change runtime type → GPU (A100 ideal; T4 ok).**

## Upload to Drive (MyDrive/)
- `pb_v4_upload.zip` — code + all 5 caches (~1.5 GB). One-time.
- `pb_v4_code.zip` — tiny code-only overlay (apply latest code fixes WITHOUT
  re-uploading the 1.5 GB bundle). Re-upload just this when code changes.

Training uses **mixed precision (AMP)** — fits a sane batch on the A100 and
runs ~2x faster. Split: train pb_3/4/5min (same court) · val pb_2min ·
**test pb_3min_court2 (different court = generalization)**.

In [ ]:
# Set BEFORE torch initializes CUDA (reduces fragmentation OOM).
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, sys
from pathlib import Path
LOCAL = Path('/content/pb_v4')
if not LOCAL.exists():
    print('unzipping bundle to local disk...')
    with zipfile.ZipFile('/content/drive/MyDrive/pb_v4_upload.zip') as z:
        z.extractall(LOCAL)
# always overlay the tiny code zip so fixes don't need a 1.5 GB re-upload
cz = Path('/content/drive/MyDrive/pb_v4_code.zip')
if cz.exists():
    with zipfile.ZipFile(cz) as z: z.extractall(LOCAL)
    print('applied code overlay')
REPO = LOCAL/'repo'; DATA = LOCAL/'data'
OUT_DRIVE = '/content/drive/MyDrive/ball_model_v4.pt'
sys.path.insert(0, str(REPO))
print('repo:', REPO.exists(), '| clips:', sorted(p.name for p in DATA.iterdir()))

In [ ]:
try:
    import cv2  # noqa
except Exception:
    !pip -q install opencv-python-headless

In [ ]:
import logging; logging.basicConfig(level=logging.INFO)
from stages.finetune_ball_model.train_v4 import TrainConfig
from stages.finetune_ball_model._v4_cache import train_from_manifests
train_folders = [DATA/'pb_3min', DATA/'pb_4min', DATA/'pb_5min']
val_folder    = DATA/'pb_2min'
test_folder   = DATA/'pb_3min_court2'
for f in train_folders+[val_folder,test_folder]:
    assert (f/'v4_manifest.json').exists(), f
print('OK — manifests present')

In [ ]:
cfg = TrainConfig(
    epochs=30,
    batch_size=8,        # A100+AMP: 8 (try 12). T4: 4. Lower if OOM.
    lr=1e-3,
    out_path=OUT_DRIVE,
)
result = train_from_manifests(train_folders, val_folder, cfg,
                              test_folder=test_folder, num_workers=4)
print('best VAL recall (same court):', result['best_val_recall'])
print('TEST recall (different court):', result['test_recall'])

## Result (bar: test recall ≥ 0.80)
- **Both high** → generalizes across courts. Done.
- **Val high, test low** → overfit to training court → add pb_3min_court2 to
  training + new court as next test (fine-tune loop).
- **Both low** → 1080p (PROC_H,PROC_W=1088,1920 in `_v4_data.py`), re-prep,
  retrain.
Download `MyDrive/ball_model_v4.pt` → `data/models/`; report both recalls.

In [ ]:
import json
rep = json.load(open('/content/drive/MyDrive/validation_report.json'))
print('val', rep['val_clip'], rep['best_val_recall'],
      '| test', rep['test_clip'], rep['test_recall_at_best_val'])
for h in rep['history'][-6:]: print(h)